# RoadWatch BD: detector benchmark on BadODD (Kaggle GPU)

The conference-paper experiment. **Question:** how much of a fine-tuned detector's BadODD score comes from
test images whose video it has already seen in training?

BadODD's official split is by image: every video has frames in train *and* test, a typical test frame only
~59 frames from a training frame. The **video split** (`ml/configs/splits/badodd_video_v1.json`, built by
`scripts/make_group_split.py`) puts each whole video in one split, with sizes and class mix matched.

| `RUNS` entry | What it does | T4 time (rough) |
|---|---|---|
| `zeroshot` | COCO RT-DETR r18 and COCO YOLO11s on both test sets (the control) | ~15 min |
| `rtdetr_official`, `rtdetr_video` | fine-tune RT-DETR r18 (Apache-2.0), 30 epochs, then evaluate | 1–2 h each |
| `yolo_official`, `yolo_video` | fine-tune YOLO11s (Ultralytics, AGPL-3.0; fine for research), 100 epochs, then evaluate | 2–3 h each |

A Kaggle session stops after 12 hours, so run it in two sessions: first `zeroshot` and both RT-DETR runs, then
both YOLO runs (edit `RUNS` in section 2).

**Before running:** Accelerator → **GPU T4**, **Internet on** (phone-verified account). Use
**Save Version → Save & Run All (Commit)** so it keeps running with the browser closed. Download
`bench_results.zip` from the **Output** tab afterwards.

Credit: BadODD, Baig et al., arXiv:2401.10659, Zenodo record 13823687, CC BY 4.0.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/temp/roadwatch-bd && git clone --depth 1 https://github.com/muttakin210/roadwatch-bd.git /kaggle/temp/roadwatch-bd
!pip install -q "transformers>=4.50" timm "torchmetrics>=1.4" "pycocotools>=2.0.8" opencv-python-headless pyyaml "ultralytics>=8.3"

## 1. Download BadODD, convert it, build both splits
`badodd_unified` keeps the official split; `badodd_video` regroups the same images by video (hard links).

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp/badodd && cd /kaggle/temp/badodd
if [ ! -d extracted ]; then
  wget -q -O badodd.zip "https://zenodo.org/api/records/13823687/files/badodd.zip/content"
  echo "6a0d3983b379302ab79a99baf7d47d74  badodd.zip" | md5sum -c -
  unzip -q -o badodd.zip -d extracted && rm badodd.zip
fi
cd /kaggle/temp/roadwatch-bd/ml
[ -d /kaggle/temp/badodd_unified ] || python scripts/convert_datasets.py --dataset badodd \
    --src /kaggle/temp/badodd/extracted --dst /kaggle/temp/badodd_unified
[ -d /kaggle/temp/badodd_video ] || python scripts/make_group_split.py build --src /kaggle/temp/badodd_unified \
    --manifest configs/splits/badodd_video_v1.json --dst /kaggle/temp/badodd_video
for d in badodd_unified badodd_video; do
  echo "$d: $(ls /kaggle/temp/$d/train/images | wc -l) train / $(ls /kaggle/temp/$d/val/images | wc -l) val / $(ls /kaggle/temp/$d/test/images | wc -l) test"
done

## 2. Choose what to run
Every score is AP@50 from `eval/eval_detector.py`, with 95 % confidence intervals that resample whole
videos (frames of one video are not independent).

In [ ]:
RUNS = ["zeroshot", "rtdetr_official", "rtdetr_video", "yolo_official", "yolo_video"]  # trim per session
EPOCHS_RTDETR, EPOCHS_YOLO, IMGSZ, SEED = 30, 100, 640, 0

import json, os, subprocess, sys
from pathlib import Path

ML = Path("/kaggle/temp/roadwatch-bd/ml")
os.chdir(ML)
sys.path.insert(0, str(ML))
from roadwatch.types import CLASSES

DATA = {"official": Path("/kaggle/temp/badodd_unified"), "video": Path("/kaggle/temp/badodd_video")}
OUT = Path("/kaggle/working/bench")
RES = OUT / "results"
RES.mkdir(parents=True, exist_ok=True)


def sh(cmd):
    # run a command, streaming its output here; stop the notebook if it fails
    cmd = [str(c) for c in cmd]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


def evaluate(split, name, *extra):
    sh([sys.executable, "eval/eval_detector.py", "--split", DATA[split] / "test",
        "--out", RES / f"{name}_{split}_test", "--bootstrap-unit", "video", *extra])

## 3. Zero-shot COCO detectors: the control
These models never saw BadODD, so their score difference between the two test sets measures how much harder
the video-split test set is, not leakage. The leakage estimate in section 6 subtracts it.

In [ ]:
if "zeroshot" in RUNS:
    for split in DATA:
        evaluate(split, "coco_rtdetr_r18")
        evaluate(split, "coco_yolo11s", "--detector", "ultralytics", "--model-id", "yolo11s.pt", "--imgsz", IMGSZ)

## 4. Fine-tune RT-DETR r18 on each split
Same recipe on both splits (`scripts/train_detector.py`); the best epoch by validation mAP@50 is kept.

In [ ]:
for split in DATA:
    if f"rtdetr_{split}" not in RUNS:
        continue
    out = OUT / f"rtdetr_r18_{split}"
    sh([sys.executable, "scripts/train_detector.py", "--data", DATA[split], "--out", out,
        "--epochs", EPOCHS_RTDETR, "--batch", 8, "--workers", 4, "--max-val", 500, "--seed", SEED])
    evaluate(split, "bd_rtdetr_r18", "--detector", "hf", "--model-id", out / "best")

## 5. Fine-tune YOLO11s on each split
Ultralytics' standard recipe (mosaic and other augmentations), same settings on both splits.

In [ ]:
from ultralytics import YOLO

for split in DATA:
    if f"yolo_{split}" not in RUNS:
        continue
    cfg = OUT / f"badodd_{split}.yaml"
    cfg.write_text(json.dumps({"path": str(DATA[split]), "train": "train/images", "val": "val/images",
                               "names": CLASSES}))
    YOLO("yolo11s.pt").train(data=str(cfg), epochs=EPOCHS_YOLO, imgsz=IMGSZ, batch=16, workers=4, seed=SEED,
                             project=str(OUT), name=f"yolo11s_{split}", exist_ok=True, plots=False)
    evaluate(split, "bd_yolo11s", "--detector", "ultralytics",
             "--model-id", OUT / f"yolo11s_{split}" / "weights" / "best.pt", "--imgsz", IMGSZ)

## 6. Results and download
**Leakage estimate** = (fine-tuned official − fine-tuned video) − (zero-shot official − zero-shot video), in mAP@50.

In [ ]:
print(f"{'result':40s} {'images':>6s} {'mAP50':>6s}  {'95% CI (videos)':15s}  mAP50, classes the model can output")
for p in sorted(RES.glob("*.json")):
    r = json.loads(p.read_text())
    lo, hi = r["map50_all_classes_ci95"]
    print(f"{p.stem:40s} {r['n_images']:6d} {r['map50_all_classes']:6.3f}  {lo:.3f}–{hi:.3f}      "
          f"{r['map50_predictable_classes']:.3f}")


def gap(prefix):
    paths = [RES / f"{prefix}_{s}_test.json" for s in DATA]
    if not all(p.exists() for p in paths):
        return None
    off, vid = (json.loads(p.read_text())["map50_all_classes"] for p in paths)
    return off - vid


print()
for det in ("rtdetr_r18", "yolo11s"):
    ft, zs = gap(f"bd_{det}"), gap(f"coco_{det}")
    if ft is not None and zs is not None:
        print(f"{det}: fine-tuned gap {ft:+.3f}, zero-shot gap {zs:+.3f} -> leakage estimate {ft - zs:+.3f} mAP@50")

In [ ]:
# results, training logs, best weights and the raw detections (.pkl, for re-scoring on your PC without a GPU)
!cd /kaggle/working && rm -f bench_results.zip && zip -qr bench_results.zip bench -x "bench/*/last/*" "bench/*/weights/last.pt" && ls -lh bench_results.zip

Unzip `bench_results.zip` into `D:\Traffic AI model project\ml\eval\kaggle\` on your PC. The `results/*.md` files
are readable as they are; the paper's tables are built from the `.json` files.